# 02. Qiskit UQK Overlap Matrix And Energy Prediction

This notebook is the Qiskit side of the workflow. Run it in the `qiskit_env_v1` environment after notebook 1 has produced a molecule-local `workflow_manifest.json`.

The workflow here is: read grouped Jordan-Wigner Pauli blocks, build grouped evolution circuits, estimate the UQK correlation sequence with MFE, assemble the Toeplitz overlap matrix, then solve the projected unitary generalized eigenvalue problem.

## Cell 1: Read The Molecule Manifest

Set `MOLECULE_NAME` to the value printed by notebook 1. Everything else is discovered from `notebooks/<molecule_name>/metadata/workflow_manifest.json`.

In [5]:
from pathlib import Path
import sys

CANDIDATE_ROOTS = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(path for path in CANDIDATE_ROOTS if (path / "notebooks" / "workflow_helpers").exists())
NOTEBOOKS_ROOT = REPO_ROOT / "notebooks"
if str(NOTEBOOKS_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_ROOT))

from workflow_helpers.qiskit_uqk_helpers import load_workflow_manifest

# --- User-facing manifest option ---
# Copy this from the final printout of notebook 1.
MOLECULE_NAME = "diatomic_h2_sto_3g"

manifest_path, manifest = load_workflow_manifest(NOTEBOOKS_ROOT, MOLECULE_NAME)


Loaded Workflow Manifest
Molecule name:                           diatomic_h2_sto_3g
Manifest JSON:                           /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/metadata/workflow_manifest.json
Workflow root:                           /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g
file: fermion_blocks_json                /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/hamiltonian_blocks/diatomic_h2_sto_3g_fermion_blocks.json
file: grouped_evolution_metadata_json    /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/circuits/diatomic_h2_sto_3g_grouped_evolution_metadata.json
file: grouped_evolution_qpy              /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/circuits/diatomic_h2_sto_3g_grouped_evolution.qpy
file: grouped_paulis_json                /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/hamiltonian_blocks/diatomic_h2_sto_3g_grouped_paulis.json
file: hermitian_pairs_json               /Us

## Cell 2: Build The UQK Overlap Matrix

This cell is the main Qiskit driver. First it builds one grouped circuit per Hermitian-pair/Jordan-Wigner block:

\[
U_\mu(\Delta t)=\exp\left[-i\Delta t\sum_\rho \alpha_{\mu\rho}P_{\mu\rho}\right].
\]

Then it estimates the correlation sequence

\[
C_k = \langle \Phi_{\mathrm{HF}} | U^k | \Phi_{\mathrm{HF}} \rangle
\]

using the MFE return-probability circuits. The standard mode composes full Trotter blocks. The stochastic mode samples qDRIFT grouped blocks. In both cases the overlap matrix is assembled by Toeplitz symmetry,

\[
S_{mn}=C_{n-m}, \qquad C_{-k}=C_k^*.
\]

The zero-body scalar phase is not measured in the MFE circuit; it is applied analytically after the non-scalar correlation is estimated. The cell writes the circuit archive, overlap matrix NPZ, metadata JSON, and updates the manifest with those paths.

In [6]:
from workflow_helpers.qiskit_uqk_helpers import (
    build_grouped_evolution_circuits,
    run_uqk_overlap_driver,
)

# --- User-facing circuit construction options ---
REBUILD_GROUPED_CIRCUITS = True
DT = 0.1
EVOLUTION_METHOD = "pauli_evolution_gate"  # "pauli_evolution_gate" or "manual_ladder"
TARGET_MODE = "generic_simulator"          # "generic_simulator" or "ibm_qpu_paid"
GENERIC_BASIS_GATES = ["rz", "sx", "x", "cx"]
TRANSPILER_OPTIMIZATION_LEVEL = 3
TROTTER_SEQUENCE_ORDER = 1
IBM_BACKEND_NAME = "ibm_brisbane"
IBM_RUNTIME_CHANNEL = "ibm_quantum_platform"
IBM_INSTANCE = None
IBM_USE_FRACTIONAL_GATES = False

# --- User-facing UQK/MFE options ---
UQK_MODE = "standard"  # "standard" or "stochastic"
BACKEND_MODE = "local_noiseless_statevector"  # "local_noiseless_statevector", "local_noisy_simple", "local_noisy_ibm_model"
KRYLOV_DIMENSION = 4
MAX_CORRELATION_POWER = KRYLOV_DIMENSION  # C_M is needed later for projected U.
SHOTS_PER_MFE_EXPERIMENT = 20_000
RANDOM_SEED = 230623
MFE_VERBOSE_FOR_FIRST_NONZERO_POWER = False

# --- User-facing stochastic qDRIFT options ---
QDRIFT_SEGMENT_COUNT_ND = 50
STOCHASTIC_INSTANCES_PER_CORRELATION = 200
STOCHASTIC_WEIGHT_CONVENTION = "group_pauli_l1_norm"

# --- User-facing noisy simulator options ---
NOISY_SIMULATION_METHOD = "density_matrix"
NOISY_TRANSPILE_OPTIMIZATION_LEVEL = 1
SIMPLE_NOISE_ONE_QUBIT_DEPOLARIZING_PROBABILITY = 1.0e-3
SIMPLE_NOISE_TWO_QUBIT_DEPOLARIZING_PROBABILITY = 1.0e-2
SIMPLE_NOISE_READOUT_ERROR_PROBABILITY = 2.0e-2
IBM_MODEL_SOURCE = "fake_backend"
IBM_MODEL_FAKE_BACKEND_CLASS = "FakeBrisbane"
IBM_MODEL_RUNTIME_BACKEND_NAME = "ibm_brisbane"
IBM_MODEL_COMPRESS_TO_ACTIVE_SPACE = True

OUTPUT_LABEL = f"{MOLECULE_NAME}_{UQK_MODE}_{BACKEND_MODE}_{SHOTS_PER_MFE_EXPERIMENT}_shots"

if REBUILD_GROUPED_CIRCUITS:
    manifest, circuit_metadata = build_grouped_evolution_circuits(
        manifest_path=manifest_path,
        manifest=manifest,
        dt=DT,
        evolution_method=EVOLUTION_METHOD,
        target_mode=TARGET_MODE,
        generic_basis_gates=GENERIC_BASIS_GATES,
        transpiler_optimization_level=TRANSPILER_OPTIMIZATION_LEVEL,
        trotter_sequence_order=TROTTER_SEQUENCE_ORDER,
        ibm_backend_name=IBM_BACKEND_NAME,
        ibm_runtime_channel=IBM_RUNTIME_CHANNEL,
        ibm_instance=IBM_INSTANCE,
        ibm_use_fractional_gates=IBM_USE_FRACTIONAL_GATES,
        verbose=True,
    )

manifest, uqk_data, uqk_metadata = run_uqk_overlap_driver(
    manifest_path=manifest_path,
    manifest=manifest,
    uqk_mode=UQK_MODE,
    backend_mode=BACKEND_MODE,
    krylov_dimension=KRYLOV_DIMENSION,
    max_correlation_power=MAX_CORRELATION_POWER,
    shots_per_mfe_experiment=SHOTS_PER_MFE_EXPERIMENT,
    qdrift_segment_count_Nd=QDRIFT_SEGMENT_COUNT_ND,
    stochastic_instances_per_correlation=STOCHASTIC_INSTANCES_PER_CORRELATION,
    stochastic_weight_convention=STOCHASTIC_WEIGHT_CONVENTION,
    random_seed=RANDOM_SEED,
    output_label=OUTPUT_LABEL,
    noisy_simulation_method=NOISY_SIMULATION_METHOD,
    noisy_transpile_optimization_level=NOISY_TRANSPILE_OPTIMIZATION_LEVEL,
    simple_noise_one_qubit_depolarizing_probability=SIMPLE_NOISE_ONE_QUBIT_DEPOLARIZING_PROBABILITY,
    simple_noise_two_qubit_depolarizing_probability=SIMPLE_NOISE_TWO_QUBIT_DEPOLARIZING_PROBABILITY,
    simple_noise_readout_error_probability=SIMPLE_NOISE_READOUT_ERROR_PROBABILITY,
    ibm_model_source=IBM_MODEL_SOURCE,
    ibm_model_fake_backend_class=IBM_MODEL_FAKE_BACKEND_CLASS,
    ibm_model_runtime_backend_name=IBM_MODEL_RUNTIME_BACKEND_NAME,
    ibm_model_compress_to_active_space=IBM_MODEL_COMPRESS_TO_ACTIVE_SPACE,
    mfe_verbose_for_first_nonzero_power=MFE_VERBOSE_FOR_FIRST_NONZERO_POWER,
)


Grouped Pauli Evolution Circuit Builder
Each saved circuit is one grouped Hermitian-pair factor:
  U_mu(dt) = exp(-i dt sum_l alpha_mu_l P_mu_l)
The full first-order Trotter step is the product of these group circuits in group-index order.

Hard-Coded User Options
Input grouped Pauli JSON:          /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/hamiltonian_blocks/diatomic_h2_sto_3g_grouped_paulis.json
Input SHA256:                      0a035674aa216e70d6ba60435c3665a675a5c8ddc51d919ab27281527b4c8a9a
Output QPY circuit archive:        /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/circuits/diatomic_h2_sto_3g_grouped_evolution.qpy
Output metadata JSON:              /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/circuits/diatomic_h2_sto_3g_grouped_evolution_metadata.json
dt:                                0.1
Evolution method keyword:          pauli_evolution_gate
Valid evolution methods:           ['manual_ladder', 'pauli_evolution_gate']
PauliE

## Cell 3: Solve The Projected Unitary Generalized Eigenvalue Problem

The UQK eigenproblem diagonalizes the projected one-step unitary, not the Hamiltonian directly:

\[
U c = \lambda S c.
\]

For an exact Hamiltonian eigenstate, \(\lambda = e^{-i E\Delta t}\), so the energy estimate is extracted from the phase:

\[
E = -\operatorname{Arg}(\lambda)/\Delta t,
\]

with the usual \(2\pi/\Delta t\) phase-wrapping ambiguity. The solver uses qforte-style canonical orthogonalization: diagonalize \(S\), discard small eigenvalues, solve the reduced ordinary eigenproblem, and back-transform the eigenvectors. The summary JSON records retained rank, condition number, eigenvalues, phases, energies, and residual diagnostics.

In [7]:
from workflow_helpers.qiskit_uqk_helpers import solve_uqk_gep_from_manifest

# --- User-facing GEP options ---
KRYLOV_DIMENSION_TO_USE = KRYLOV_DIMENSION
KRYLOV_DT = DT
OVERLAP_EIGENVALUE_THRESHOLD = None  # None mirrors qforte's default 1e-15 behavior.
SORT_ROOTS_BY = "energy_real_ascending"
USE_DIRECT_VALIDATION_WHEN_AVAILABLE = True
OUTPUT_SUMMARY_NAME = f"{OUTPUT_LABEL}_uqk_gep_summary.json"

manifest, gep_summary = solve_uqk_gep_from_manifest(
    manifest_path=manifest_path,
    manifest=manifest,
    krylov_dimension_to_use=KRYLOV_DIMENSION_TO_USE,
    dt=KRYLOV_DT,
    overlap_eigenvalue_threshold=OVERLAP_EIGENVALUE_THRESHOLD,
    sort_roots_by=SORT_ROOTS_BY,
    output_summary_name=OUTPUT_SUMMARY_NAME,
    use_direct_validation_when_available=USE_DIRECT_VALIDATION_WHEN_AVAILABLE,
)


Projected Unitary UQK GEV Solver
This script reads a saved UQK correlation sequence C_k, assembles
S_mn=C_(n-m) and U_mn=C_(n+1-m), solves U c=lambda S c by canonical
orthogonalization of S, then converts eigenphases to energies.

Hard-Coded Options
Input correlation NPZ:                 /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/results/h4_diatomic_h2_sto_3g_standard_local_noiseless_statevector_20000_shots_uqk_overlap_matrix.npz
Input correlation metadata:            /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/results/h4_diatomic_h2_sto_3g_standard_local_noiseless_statevector_20000_shots_uqk_overlap_matrix_metadata.json
Input molecule metadata:               /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/metadata/diatomic_h2_sto_3g_metadata.json
Direct validation NPZ:                 /Users/nstair/Src/qforte-qiskit/notebooks/diatomic_h2_sto_3g/summaries/direct_validation_optional.npz
Krylov dimension M:                    4
dt:        